# Machine Learning Dataset Preparation

This notebook prepares a clean supervised-learning dataset for the Vietnamese ETF Decision Support System. It does not train a model and does not write to PostgreSQL.

Target definition: `future_return_5d = close.shift(-5) / close - 1`, calculated separately for each ETF. The binary target is `1` when the future 5-trading-day return is positive and `0` otherwise.

## 1. Setup

Use pandas, numpy, SQLAlchemy, and the shared backend feature-building code. Database credentials are loaded from the existing backend configuration.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_ROOT = PROJECT_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.core.config import settings
from app.ml.features import (
    FEATURE_COLUMNS,
    build_ml_dataset,
    load_ml_source_data,
    purged_chronological_split,
    validate_purged_split,
)

engine = create_engine(settings.database_url, pool_pre_ping=True)
FEATURE_COLUMNS

['daily_return',
 'volume_change',
 'close_to_sma20',
 'close_to_sma50',
 'rsi_14',
 'macd',
 'macd_signal',
 'volatility_20',
 'macd_histogram']

## 2. Load Source Data

Load joined price history and technical indicator data from PostgreSQL. This is read-only and uses `price_history` joined to `technical_indicators` by ETF and date.

In [2]:
source_df = load_ml_source_data(engine)
source_df.head()

,symbol,date,close,volume,sma_20,sma_50,rsi_14,macd,macd_signal,volatility_20
0,E1VFVN30,2022-01-04,26.20,959300,NaN,NaN,NaN,0.000000,0.000000,NaN
1,E1VFVN30,2022-01-05,25.99,1500400,NaN,NaN,NaN,-0.016752,-0.003350,NaN
2,E1VFVN30,2022-01-06,25.82,1612600,NaN,NaN,NaN,-0.043247,-0.011330,NaN
3,E1VFVN30,2022-01-07,25.85,1417700,NaN,NaN,NaN,-0.061120,-0.021288,NaN
4,E1VFVN30,2022-01-10,25.58,1313400,NaN,NaN,NaN,-0.095964,-0.036223,NaN


## 3. Build ML Dataset

The dataset builder calculates features using only information available at time `t`. The only future-looking operation is `close.shift(-5)`, and it is used only to create the target, not as a feature. Rows without enough future data or enough indicator lookback data are removed.

In [3]:
ml_df = build_ml_dataset(source_df, horizon_days=5)
ml_df.head()

,symbol,date,daily_return,volume_change,close_to_sma20,close_to_sma50,rsi_14,macd,macd_signal,volatility_20,macd_histogram,future_return_5d,target,target_date
0,E1VFVN30,2022-03-21,0.011200,-0.316536,-0.000751,-0.009226,44.244604,-0.208241,-0.196381,0.146541,-0.011860,-0.005934,0,2022-03-28
1,E1VFVN30,2022-03-22,0.005142,9.111700,0.005361,-0.003514,50.184502,-0.167913,-0.190687,0.148213,0.022774,-0.010626,0,2022-03-29
2,E1VFVN30,2022-03-23,-0.005510,-0.844213,0.000832,-0.008444,42.023346,-0.145571,-0.181664,0.148537,0.036093,-0.004749,0,2022-03-30
3,E1VFVN30,2022-03-24,-0.004749,-0.896020,-0.003428,-0.012634,39.700375,-0.135981,-0.172527,0.140243,0.036546,0.005964,1,2022-03-31
4,E1VFVN30,2022-03-25,0.001193,11.010782,-0.001606,-0.010935,41.923077,-0.124524,-0.162927,0.139439,0.038402,0.027800,1,2022-04-01


## 4. Dataset Shape and Coverage

Summarize the dataset size, rows per ETF, and date range after removing rows that cannot be used for supervised learning.

In [4]:
print("Dataset shape:", ml_df.shape)

rows_per_etf = ml_df.groupby("symbol").size().reset_index(name="rows")
date_coverage = ml_df.groupby("symbol").agg(min_date=("date", "min"), max_date=("date", "max")).reset_index()

display(rows_per_etf)
display(date_coverage)

Dataset shape: (5302, 14)


,symbol,rows
0,E1VFVN30,1096
1,FUEDCMID,918
2,FUESSVFL,1096
3,FUEVFVND,1096
4,FUEVN100,1096


,symbol,min_date,max_date
0,E1VFVN30,2022-03-21,2026-08-14
1,FUEDCMID,2022-12-07,2026-08-14
2,FUESSVFL,2022-03-21,2026-08-14
3,FUEVFVND,2022-03-21,2026-08-14
4,FUEVN100,2022-03-21,2026-08-14


## 5. Target Class Distribution

Check whether the binary target is balanced overall and within each ETF.

In [5]:
target_distribution = ml_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="rows")
target_distribution["percentage"] = target_distribution["rows"] / target_distribution["rows"].sum()
target_distribution

,target,rows,percentage
0,0,2401,0.452848
1,1,2901,0.547152


In [6]:
target_by_etf = (
    ml_df.groupby(["symbol", "target"])
    .size()
    .reset_index(name="rows")
)
target_by_etf["percentage"] = target_by_etf["rows"] / target_by_etf.groupby("symbol")["rows"].transform("sum")
target_by_etf

,symbol,target,rows,percentage
0,E1VFVN30,0,486,0.443431
1,E1VFVN30,1,610,0.556569
2,FUEDCMID,0,410,0.446623
3,FUEDCMID,1,508,0.553377
4,FUESSVFL,0,497,0.453467
5,FUESSVFL,1,599,0.546533
6,FUEVFVND,0,514,0.468978
7,FUEVFVND,1,582,0.531022
8,FUEVN100,0,494,0.450730
9,FUEVN100,1,602,0.549270


## 6. Feature Statistics and Quality Checks

Inspect descriptive statistics, missing values, and infinite values for model input features.

In [7]:
ml_df[FEATURE_COLUMNS].describe()

,daily_return,volume_change,close_to_sma20,close_to_sma50,rsi_14,macd,macd_signal,volatility_20,macd_histogram
count,5302.000000,5302.000000,5302.000000,5302.000000,5302.000000,5302.000000,5302.000000,5302.000000,5302.000000
mean,0.000377,1.264012,0.002824,0.007555,54.008979,0.040359,0.040855,0.214734,-0.000496
std,0.015183,8.987324,0.035309,0.058769,18.297236,0.365559,0.345628,0.114159,0.105499
min,-0.091892,-0.992718,-0.208355,-0.281998,2.083333,-1.342958,-1.064712,0.041946,-0.663942
25%,-0.005470,-0.477351,-0.015261,-0.021643,40.501510,-0.124522,-0.113259,0.131147,-0.053368
50%,0.000697,0.008285,0.006221,0.010116,53.138572,0.056133,0.053065,0.184131,0.007635
75%,0.007217,0.908845,0.024139,0.043374,67.402704,0.229402,0.216791,0.262326,0.055894
max,0.069953,385.687500,0.164800,0.291703,100.000000,2.011429,1.771512,0.739901,0.492120


In [8]:
missing_values = ml_df[FEATURE_COLUMNS + ["future_return_5d", "target"]].isna().sum()
infinite_values = np.isinf(ml_df[FEATURE_COLUMNS + ["future_return_5d"]]).sum()

display(missing_values.rename("missing_count"))
display(infinite_values.rename("infinite_count"))

daily_return        0
volume_change       0
close_to_sma20      0
close_to_sma50      0
rsi_14              0
macd                0
macd_signal         0
volatility_20       0
macd_histogram      0
future_return_5d    0
target              0
Name: missing_count, dtype: int64

daily_return        0
volume_change       0
close_to_sma20      0
close_to_sma50      0
rsi_14              0
macd                0
macd_signal         0
volatility_20       0
macd_histogram      0
future_return_5d    0
Name: infinite_count, dtype: int64

## 7. Feature Correlation

Review correlations between numeric input features. This is diagnostic only; no feature selection or model training is performed here.

In [9]:
feature_correlation = ml_df[FEATURE_COLUMNS].corr()
feature_correlation

,daily_return,volume_change,close_to_sma20,close_to_sma50,rsi_14,macd,macd_signal,volatility_20,macd_histogram
daily_return,1.000000,-0.030539,0.377056,0.238282,0.238602,0.068706,0.020395,0.003712,0.171251
volume_change,-0.030539,1.000000,0.005655,-0.000106,0.005680,-0.002539,-0.006975,0.022657,0.014054
close_to_sma20,0.377056,0.005655,1.000000,0.820898,0.830732,0.709211,0.515801,-0.203270,0.767617
close_to_sma50,0.238282,-0.000106,0.820898,1.000000,0.688250,0.915405,0.859158,-0.382208,0.357209
rsi_14,0.238602,0.005680,0.830732,0.688250,1.000000,0.638888,0.471444,-0.219323,0.669264
macd,0.068706,-0.002539,0.709211,0.915405,0.638888,1.000000,0.957527,-0.376396,0.328066
macd_signal,0.020395,-0.006975,0.515801,0.859158,0.471444,0.957527,1.000000,-0.399764,0.041746
volatility_20,0.003712,0.022657,-0.203270,-0.382208,-0.219323,-0.376396,-0.399764,1.000000,0.005447
macd_histogram,0.171251,0.014054,0.767617,0.357209,0.669264,0.328066,0.041746,0.005447,1.000000


## 8. Purged Chronological Train/Test Split Design

For financial time-series data, the split must be chronological. Random `train_test_split` is inappropriate because it can mix future observations into the training set and create leakage.

This dataset predicts a 5-trading-period future return. A normal chronological split is still not enough: training rows immediately before the test cutoff may use target prices that occur inside the test period. The split below applies one common test start date, then purges training rows whose `target_date` is on or after the test start date. The purge uses ETF-specific trading-observation order through the precomputed `target_date`; it does not subtract calendar days.

In [10]:
unique_dates = pd.Series(sorted(ml_df["date"].unique()))
cutoff_index = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates.iloc[cutoff_index]

train_df, test_df, purged_df = purged_chronological_split(
    ml_df,
    test_start_date=cutoff_date,
    horizon=5,
)
validate_purged_split(train_df, test_df, test_start_date=cutoff_date)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": len(train_df),
            "min_date": train_df["date"].min(),
            "max_date": train_df["date"].max(),
        },
        {
            "split": "purged",
            "row_count": len(purged_df),
            "min_date": purged_df["date"].min(),
            "max_date": purged_df["date"].max(),
        },
        {
            "split": "test",
            "row_count": len(test_df),
            "min_date": test_df["date"].min(),
            "max_date": test_df["date"].max(),
        },
    ]
)

train_target_distribution = train_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="train_rows")
test_target_distribution = test_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="test_rows")

print("Original cutoff / test start date:", cutoff_date)
display(split_summary)
display(train_target_distribution)
display(test_target_distribution)

Original cutoff / test start date: 2025-09-29 00:00:00


,split,row_count,min_date,max_date
0,train,4177,2022-03-21,2025-09-19
1,purged,25,2025-09-22,2025-09-26
2,test,1100,2025-09-29,2026-08-14


,target,train_rows
0,0,1811
1,1,2366


,target,test_rows
0,0,580
1,1,520
